In [3]:
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.exceptions import ConvergenceWarning
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore", category=ConvergenceWarning)  # LogReg sin escalar no converge (es esperado)
pd.set_option("display.max_columns", None)

In [4]:
columns = ["age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week",
    "native_country", "income"]

df = pd.read_csv(
    "../data/raw/adult.data",
    names=columns,
    skipinitialspace=True,   # quita el espacio antes de cada valor
    na_values="?"            # ' ?' -> NaN
)

print(df.shape)
print("\nNulos por columna:")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nTarget:")
print(df["income"].value_counts())
df.head()

(32561, 15)

Nulos por columna:
workclass         1836
occupation        1843
native_country     583
dtype: int64

Target:
income
<=50K    24720
>50K      7841
Name: count, dtype: int64


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


### 1. Construimos X e y (sin leakage)

In [5]:
# ---------------- JUSTIFICACIÓN ANTI-LEAKAGE ----------------
# Escenario: predecir si una persona gana >50K al año usando datos que
# ella misma podría darnos en un formulario ANTES de conocer su ingreso.
#
# Features usadas (todas conocidas al momento de predecir):
#   - age, sex, race, native_country       -> datos demográficos fijos
#   - education_num                         -> nivel educativo alcanzado
#   - marital_status, relationship          -> situación familiar actual
#   - workclass, occupation, hours_per_week -> situación laboral actual
# Ninguna se calcula a partir del ingreso ni se conoce "después" de él.
#
# Features EXCLUIDAS:
#   - income                     -> es el target
#   - fnlwgt                     -> peso muestral del censo; lo calcula la oficina
#                                   de estadística después de la encuesta, no es
#                                   un atributo de la persona
#   - education                  -> duplica education_num (mismo dato en texto)
#   - capital_gain, capital_loss -> se realizan en el mismo año que el ingreso y
#                                   forman parte de él; usarlas sería meterle al
#                                   modelo parte de la respuesta
# -------------------------------------------------------------

num_cols = ["age", "education_num", "hours_per_week"]
cat_cols = ["workclass", "marital_status", "occupation", "relationship",
            "race", "sex", "native_country"]

X = df[num_cols + cat_cols].copy()
y = (df["income"] == ">50K").astype(int)   # clase positiva = 1 (>50K)

print("X:", X.shape)
print("Proporción clase positiva:", round(y.mean(), 3))

X: (32561, 10)
Proporción clase positiva: 0.241


### 2. Split estratificado y baseline Dummy

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,        # mantiene la proporción de >50K en train y test
    random_state=42
)

print("Train:", X_train.shape, "| % positivos:", round(y_train.mean(), 3))
print("Test: ", X_test.shape,  "| % positivos:", round(y_test.mean(), 3))

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
pred = dummy.predict(X_test)

print("\nBaseline Dummy (siempre predice <=50K):")
print("  Accuracy test:", round(accuracy_score(y_test, pred), 3))
print("  F1 clase positiva test:", round(f1_score(y_test, pred, zero_division=0), 3))

Train: (26048, 10) | % positivos: 0.241
Test:  (6513, 10) | % positivos: 0.241

Baseline Dummy (siempre predice <=50K):
  Accuracy test: 0.759
  F1 clase positiva test: 0.0


### 3. Preprocesamiento y modelos dentro de Pipelines

In [7]:
def make_preprocessor(scale):
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num_steps), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),  # rellena los NaN que eran '?'
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), cat_cols),
    ])

models = {
    "Dummy":               (DummyClassifier(strategy="most_frequent"), False),
    "LogReg sin escalar":  (LogisticRegression(max_iter=200), False),
    "LogReg escalado":     (LogisticRegression(max_iter=1000), True),
    "Árbol (sin límite)":  (DecisionTreeClassifier(random_state=42), False),
    "Árbol (max_depth=2)": (DecisionTreeClassifier(max_depth=2, random_state=42), False),
    "Random Forest":       (RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1), False),
    "XGBoost":             (XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                                          eval_metric="logloss", random_state=42, n_jobs=-1), False),
    "LightGBM":            (LGBMClassifier(n_estimators=300, learning_rate=0.1,
                                           random_state=42, verbose=-1), False),
}

pipelines = {
    name: Pipeline([("prep", make_preprocessor(scale)), ("model", model)])
    for name, (model, scale) in models.items()
}

list(pipelines.keys())

['Dummy',
 'LogReg sin escalar',
 'LogReg escalado',
 'Árbol (sin límite)',
 'Árbol (max_depth=2)',
 'Random Forest',
 'XGBoost',
 'LightGBM']

### 4. Entrenar y armar la tabla comparativa

In [8]:
rows = []
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    pred_tr = pipe.predict(X_train)
    pred_te = pipe.predict(X_test)
    rows.append({
        "modelo": name,
        "acc_train": accuracy_score(y_train, pred_tr),
        "acc_test": accuracy_score(y_test, pred_te),
        "f1_pos_train": f1_score(y_train, pred_tr, zero_division=0),
        "f1_pos_test": f1_score(y_test, pred_te, zero_division=0),
    })
    print(f"✓ {name}")

results = pd.DataFrame(rows).set_index("modelo")
results["gap_f1"] = results["f1_pos_train"] - results["f1_pos_test"]
results.round(3).sort_values("f1_pos_test", ascending=False)

✓ Dummy
✓ LogReg sin escalar
✓ LogReg escalado
✓ Árbol (sin límite)
✓ Árbol (max_depth=2)
✓ Random Forest
✓ XGBoost
✓ LightGBM


,acc_train,acc_test,f1_pos_train,f1_pos_test,gap_f1
modelo,,,,,
XGBoost,0.868,0.843,0.708,0.650,0.057
LightGBM,0.878,0.841,0.731,0.647,0.084
LogReg sin escalar,0.835,0.835,0.619,0.620,-0.001
LogReg escalado,0.835,0.833,0.621,0.616,0.005
Random Forest,0.970,0.816,0.937,0.598,0.339
Árbol (sin límite),0.970,0.783,0.935,0.535,0.400
Árbol (max_depth=2),0.819,0.819,0.522,0.522,0.001
Dummy,0.759,0.759,0.000,0.000,0.000


### 5. Detectar sobreajuste y subajuste

In [9]:
sin_dummy = results.drop(index="Dummy")

sobre = sin_dummy["gap_f1"].idxmax()        # mayor diferencia train - test
sub = sin_dummy["f1_pos_train"].idxmin()    # ni siquiera aprende bien train

r = results.round(3)
print(f"SOBREAJUSTA: {sobre}")
print(f"  F1 train = {r.loc[sobre, 'f1_pos_train']} | F1 test = {r.loc[sobre, 'f1_pos_test']} "
      f"| gap = {r.loc[sobre, 'gap_f1']}")
print(f"  -> Memoriza train y generaliza peor.\n")

print(f"SUBAJUSTA: {sub}")
print(f"  F1 train = {r.loc[sub, 'f1_pos_train']} | F1 test = {r.loc[sub, 'f1_pos_test']} "
      f"| gap = {r.loc[sub, 'gap_f1']}")
print(f"  -> Rinde bajo incluso en train: el modelo es demasiado simple.")

SOBREAJUSTA: Árbol (sin límite)
  F1 train = 0.935 | F1 test = 0.535 | gap = 0.4
  -> Memoriza train y generaliza peor.

SUBAJUSTA: Árbol (max_depth=2)
  F1 train = 0.522 | F1 test = 0.522 | gap = 0.001
  -> Rinde bajo incluso en train: el modelo es demasiado simple.


## Conclusiones

- **Baseline:** el Dummy tiene accuracy ≈ 0.76 pero F1 de la clase positiva = 0, porque nunca predice >50K. Por eso la accuracy sola engaña: usamos el F1 de la clase positiva.
- **Sobreajuste:** el Árbol sin límite obtiene F1 train = 0.935 y F1 test = 0.535 (gap = 0.400). Crece hasta memorizar cada ejemplo de train.
- **Subajuste:** el Árbol con max_depth=2 obtiene F1 train = 0.522 y F1 test = 0.522. Rinde bajo en ambos y el gap es pequeño: el modelo es demasiado simple.
- **Escalado:** LogReg escalado vs sin escalar: 0.616 vs 0.620. Sin escalar, el optimizador rindió mejor.
- **Mejor modelo:** XGBoost con F1 test = 0.650.